# 3_HH_HM_group_patterns

Combined operation-group and entity-group HH/HM structural comparison for the matched 7/8B model set.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

_NOTEBOOK_DIR = Path.cwd()
if (_NOTEBOOK_DIR / "helpers.py").exists():
    sys.path.insert(0, str(_NOTEBOOK_DIR.parent))
elif (_NOTEBOOK_DIR / "notebooks" / "helpers.py").exists():
    sys.path.insert(0, str(_NOTEBOOK_DIR))

from notebooks.helpers import (
    ROOT,
    LATEX_TABLES,
    pretty_print_path,
    hh_question_means,
    hm_question_means,
    variant_summary_table,
    pairwise_variant_correlation_table,
    grouped_pattern_table,
    scatter_correlation_table,
    blind_accuracy_summary,
    qualitative_qdf,
    attach_answer_summaries,
    hh_ranked_examples,
    variant_top_bottom_table,
    hh_degradation_table,
    top_questions_tables,
    to_latex_table,
)

from notebooks.helpers import MATCHED_7B_MODEL_GROUP


In [ ]:
op_corr = grouped_pattern_table("op")
display(op_corr)

out = LATEX_TABLES / "group_pattern_correlation_operation.tex"
to_latex_table(
    op_corr,
    out,
    "Operation-group correlation between grouped HH and grouped HM SBERT (matched 7/8B models).",
    "tab:group_pattern_corr_operation",
    float_formatters={"Pearson r": ".3f", "Pearson p": ".1e", "Spearman rho": ".3f", "Spearman p": ".1e"},
)
print(pretty_print_path(out))

## Operation Conclusions

This table is most useful for comparing how strongly each model preserves the human ordering across operation types.

In [ ]:
op_group_summary = (
    op_corr.assign(Group=op_corr["Model"].map(MATCHED_7B_MODEL_GROUP))
    .groupby("Group", as_index=False)[["Pearson r", "Spearman rho"]]
    .mean()
    .sort_values("Pearson r", ascending=False)
)
display(op_group_summary)

best_op_group = op_group_summary.iloc[0]["Group"]
worst_op_group = op_group_summary.iloc[-1]["Group"]
print(f"Operation-level conclusion: {best_op_group} preserves the human pattern most strongly on average, while {worst_op_group} is weakest.")
print("Interpretation: operation-type structure is where architecture-level differences are easiest to see, especially for the decoder vs standalone contrast.")

In [ ]:
entity_corr = grouped_pattern_table("ent")
display(entity_corr)

out = LATEX_TABLES / "group_pattern_correlation_entity.tex"
to_latex_table(
    entity_corr,
    out,
    "Entity-group correlation between grouped HH and grouped HM SBERT (matched 7/8B models).",
    "tab:group_pattern_corr_entity",
    float_formatters={"Pearson r": ".3f", "Pearson p": ".1e", "Spearman rho": ".3f", "Spearman p": ".1e"},
)
print(pretty_print_path(out))

## Entity Conclusions

Entity-group correlations provide the parallel view for semantic fields rather than reasoning operations.

In [ ]:
entity_group_summary = (
    entity_corr.assign(Group=entity_corr["Model"].map(MATCHED_7B_MODEL_GROUP))
    .groupby("Group", as_index=False)[["Pearson r", "Spearman rho"]]
    .mean()
    .sort_values("Pearson r", ascending=False)
)
display(entity_group_summary)

best_ent_group = entity_group_summary.iloc[0]["Group"]
worst_ent_group = entity_group_summary.iloc[-1]["Group"]
print(f"Entity-level conclusion: {best_ent_group} aligns best with the human ordering on average, while {worst_ent_group} is weakest.")
print("Interpretation: entity-group structure is still informative, but the operation-group view is usually the cleaner summary of human–model pattern alignment.")

## HH SBERT Boxplots by Operation and Entity (across variants)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import wilcoxon
from pathlib import Path

sys.path.insert(0, str(ROOT))
from figures.helpers import load_cleaned_pair_cache
from utils.constants import VARIANT_COLORS

EXPORTS = ROOT / "analysis" / "session2" / "exports"

# ── Constants ────────────────────────────────────────────────────────────────
VARIANT_NAME = {"C": "Original", "B": "Weaker", "A": "Pronominalized"}
VARIANT_ORDER_LONG = ["Original", "Weaker", "Pronominalized"]
VARIANT_PALETTE = {
    "Original":      VARIANT_COLORS["C"],
    "Weaker":        VARIANT_COLORS["B"],
    "Pronominalized":VARIANT_COLORS["A"],
}

OP_MERGE = {
    "attr": "attr", "count": "count", "ident": "ident", "spat": "spat",
    "exist": "exist", "act": "act", "know": "other", "text": "other",
    "temp": "other", "comp": "other", "other": "other", "cause": "other",
}
OP_GROUP_ORDER  = ["attr", "count", "spat", "ident", "act", "exist", "other"]
OP_GROUP_LABELS = {
    "attr": "Attribute", "count": "Counting", "spat": "Spatial Relation",
    "ident": "Identity", "act": "Action", "exist": "Existence", "other": "Other",
}

ENT_MERGE = {
    "person": "person", "animal": "animal", "object": "object",
    "food": "food", "other": "other", "product": "other",
    "place": "other", "vehicle": "other", "text": "other",
}
ENT_ORDER  = ["person", "animal", "object", "food", "other"]
ENT_LABELS = {"person": "Person", "animal": "Animal", "object": "Object",
               "food": "Food", "other": "Other"}

# ── Data loading ─────────────────────────────────────────────────────────────
pair_df = load_cleaned_pair_cache(ROOT, condition="inst_blind", include_yesno=True, verbose=True)
hh = pair_df[pair_df["pair_type"] == "HH"].copy()

human = pd.read_csv(EXPORTS / "responses_human.csv")
meta = (
    human[human["variant"] == "C"]
    .drop_duplicates("question_id")[["question_id", "ent", "op"]]
    .copy()
)
meta["ent_group"] = meta["ent"].map(ENT_MERGE).fillna("other")

rows = []
for v_code, v_name in VARIANT_NAME.items():
    hh_v = (
        hh[hh["variant"] == v_code]
        .groupby("question_id", as_index=False)["sbert_score"]
        .mean()
        .rename(columns={"sbert_score": "hh_sbert"})
    )
    tmp = hh_v.merge(meta, on="question_id", how="left")
    tmp["variant_name"] = v_name
    rows.append(tmp)

long_df = pd.concat(rows, ignore_index=True)
long_df["op_group"] = long_df["op"].map(OP_MERGE).fillna("other")
long_df = long_df[
    long_df["op_group"].isin(OP_GROUP_ORDER) & long_df["ent_group"].isin(ENT_ORDER)
].copy()
long_df["op_label"]  = long_df["op_group"].map(OP_GROUP_LABELS)
long_df["ent_label"] = long_df["ent_group"].map(ENT_LABELS)

print(f"long_df: {len(long_df)} rows")

In [ ]:
# ── Helper functions (mirrored from figures/hh/analysis.py) ──────────────────

def _paired_ca_pvalues(long_df, group_col, order):
    out = []
    sub = long_df[long_df["variant_name"].isin(["Original", "Pronominalized"])].copy()
    for label in order:
        grp = sub[sub[group_col] == label]
        wide = (
            grp.pivot_table(index="question_id", columns="variant_name",
                            values="hh_sbert", aggfunc="mean")
            .dropna(subset=["Original", "Pronominalized"])
        )
        if len(wide) < 3:
            continue
        _, p = wilcoxon(wide["Original"], wide["Pronominalized"])
        out.append((label, float(p) if np.isfinite(p) else np.nan))
    return out


def _sig_text(p):
    if not np.isfinite(p):
        return None
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return None


def _annotate_significance(ax, long_df, x_col, order):
    pairs = _paired_ca_pvalues(long_df, x_col, order)
    sig_pairs = [(label, p) for label, p in pairs if _sig_text(p) is not None]
    if not sig_pairs:
        return
    sig_color = "black"
    base, step, half_width = 1.04, 0.075, 0.30
    for i, (label, p) in enumerate(sig_pairs):
        idx = order.index(label)
        x0, x1 = idx - half_width, idx + half_width
        y = base + i * step
        ax.plot([x0, x0, x1, x1], [y - 0.020, y, y, y - 0.020],
                color=sig_color, lw=2.5, clip_on=False)
        ax.text(idx, y + 0.012, _sig_text(p),
                ha="center", va="bottom", fontsize=15,
                fontweight="bold", color=sig_color, clip_on=False)


def draw_boxplot(ax, long_df, x_col, order, *, show_legend):
    sns.boxplot(data=long_df, x=x_col, y="hh_sbert", hue="variant_name",
                order=order, hue_order=VARIANT_ORDER_LONG, palette=VARIANT_PALETTE,
                width=0.56, fliersize=2.2, linewidth=0.9, ax=ax)
    ax.set_ylabel("")
    ax.set_xlabel("")
    ax.grid(axis="y", alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xticklabels(order, rotation=0, fontsize=12.5)
    ax.tick_params(axis="y", labelsize=13)
    leg = ax.get_legend()
    if not show_legend:
        if leg is not None:
            leg.remove()
    else:
        ax.legend(title=None, ncol=3, loc="upper center",
                  bbox_to_anchor=(0.5, -0.20), frameon=False, fontsize=10.5)


# ── Operation boxplot ─────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", context="talk")

op_df = long_df[long_df["op_group"].isin(OP_GROUP_ORDER)].copy()
op_order = [OP_GROUP_LABELS[k] for k in OP_GROUP_ORDER if OP_GROUP_LABELS[k] in op_df["op_label"].unique()]

fig_op, ax_op = plt.subplots(figsize=(7.3, 3.2))
draw_boxplot(ax_op, op_df, "op_label", op_order, show_legend=False)
_annotate_significance(ax_op, op_df, "op_label", op_order)
ax_op.set_ylim(None, 1.0)
ax_op.set_ylabel("HH SBERT", fontsize=13)
ax_op.tick_params(axis="x", labelsize=13)
plt.setp(ax_op.get_xticklabels(), rotation=20, ha="right")
handles, labels = ax_op.get_legend_handles_labels()
if handles:
    fig_op.legend(handles, labels, title=None, ncol=3,
                  loc="upper center", bbox_to_anchor=(0.5, 0.90),
                  frameon=False, fontsize=13, columnspacing=1.2, handletextpad=0.5)
fig_op.tight_layout(rect=[0, 0, 1, 0.91])
plt.show()

# ── Entity boxplot ────────────────────────────────────────────────────────────
ent_df = long_df[long_df["ent_group"].isin(ENT_ORDER)].copy()
ent_order = (
    ent_df[ent_df["variant_name"] == "Original"]
    .groupby("ent_label")["hh_sbert"].mean()
    .sort_values(ascending=False)
    .index.tolist()
)

fig_ent, ax_ent = plt.subplots(figsize=(6.5, 3.0))
draw_boxplot(ax_ent, ent_df, "ent_label", ent_order, show_legend=False)
_annotate_significance(ax_ent, ent_df, "ent_label", ent_order)
ax_ent.set_ylim(None, 1.0)
ax_ent.set_ylabel("HH SBERT", fontsize=13)
ax_ent.tick_params(axis="x", labelsize=13)
plt.setp(ax_ent.get_xticklabels(), rotation=20, ha="right")
fig_ent.tight_layout()
plt.show()